# What Factors Influence What Mitigation Action is Taken Once a Delay Occurs?

### Author
Imran Qureshi

### Notebook Overview
This notebook focuses specifically on predicting mitigation actions taken during shipment disruptions using classification-based analytical techniques in R. The analysis evaluates operational, transportation-related, route-related, and disruption-related variables associated with corrective supply chain responses.


# Exploratory Data Analysis (EDA)

Initial exploratory data analysis was performed to better understand the distributions and variation of shipment-related variables before predictive modeling.


In [ ]:
library(caret)
library(kknn)
library(rpart)
library(rpart.plot)
library(skimr)
library(psych)

df <- read.csv("C:/Users/iqure/Downloads/IT 190 Project - Imran Qureshi - Data.csv")
str(df)

df <- df[df$Delivery_Status == "Late", ]
df$Shipping_Cost_USD <- as.numeric(gsub("[^0-9.]", "", df$Shipping_Cost_USD))
df$Cost..USD....per.Kg <- as.numeric(gsub("[^0-9.]", "", df$Cost..USD....per.Kg))
df$Order_Date <- as.Date(df$Order_Date, format = "%m/%d/%Y")

df$Mitigation_Action_Taken <- factor(df$Mitigation_Action_Taken)
df$Route_Type <- factor(df$Route_Type)
df$Product_Category <- factor(df$Product_Category)
df$Transportation_Mode <- factor(df$Transportation_Mode)
df$Disruption_Event <- factor(df$Disruption_Event)

summary(df)
skim(df)
psych::describe(df)

par(mfrow=c(2,2))
hist(df$Geopolitical_Risk_Index, main = "Geo")
hist(df$Weather_Severity_Index, main = "Weather")
hist(df$Inflation_Rate_Pct, main = "Inflation")
hist(df$Cost..USD....per.Kg, main = "Shipping Cost per kg")
dev.off()


Initial exploratory analysis showed that mitigation actions varied substantially across different shipment disruptions and operational conditions within the dataset.


# k-Nearest Neighbors (kNN)

A k-nearest neighbors classification model was developed to predict mitigation actions taken during shipment disruptions. Numerical variables were normalized prior to training to ensure equal weighting during distance calculations.


In [ ]:
set.seed(123)
train_id <- createDataPartition(df$Mitigation_Action_Taken, p = 0.7, list = FALSE)
df.train <- df[train_id, ]
df.test  <- df[-train_id, ]

df.train$Cost..USD....per.Kg <- log(df.train$Cost..USD....per.Kg + 1)
df.test$Cost..USD....per.Kg  <- log(df.test$Cost..USD....per.Kg  + 1)

zscore <- preProcess(
  df.train[, c("Geopolitical_Risk_Index", "Weather_Severity_Index",
               "Inflation_Rate_Pct", "Cost..USD....per.Kg", "Delay_Days")],
  method = c("center", "scale")
)

norm_cols <- c("Geopolitical_Risk_Index.n", "Weather_Severity_Index.n",
               "Inflation_Rate_Pct.n", "Delay_Days.n", "Cost..USD....per.Kg.n")

df.train[norm_cols] <- predict(
  zscore, df.train[, c("Geopolitical_Risk_Index", "Weather_Severity_Index",
                       "Inflation_Rate_Pct", "Delay_Days", "Cost..USD....per.Kg")]
)

df.test[norm_cols] <- predict(
  zscore, df.test[, c("Geopolitical_Risk_Index", "Weather_Severity_Index",
                      "Inflation_Rate_Pct", "Delay_Days", "Cost..USD....per.Kg")]
)

k.value <- 5
disfun  <- 2
comfun  <- "rectangular"

base.knn <- kknn(
  Mitigation_Action_Taken ~ Geopolitical_Risk_Index.n + Weather_Severity_Index.n +
    Inflation_Rate_Pct.n + Cost..USD....per.Kg.n + Delay_Days.n + Route_Type + Product_Category + Transportation_Mode + Disruption_Event,
  train    = df.train,
  test     = df.test,
  k        = k.value,
  distance = disfun,
  kernel   = comfun
)

df.test$pred.knn.base <- fitted(base.knn)

table(Predicted = df.test$pred.knn.base, Actual = df.test$Mitigation_Action_Taken)
accuracy.knn.base <- mean(df.test$pred.knn.base == df.test$Mitigation_Action_Taken)
cat("Base kNN Accuracy:", accuracy.knn.base, "\n")

confusionMatrix(df.test$pred.knn.base, df.test$Mitigation_Action_Taken)

ctrl <- trainControl(method = "cv", number = 5)

tunegrid <- expand.grid(
  kmax     = seq(5, 25, by = 2),
  distance = c(2),
  kernel   = c("rectangular")
)

set.seed(42)
cv.fit.knn <- train(
  Mitigation_Action_Taken ~ Geopolitical_Risk_Index.n + Weather_Severity_Index.n +
    Inflation_Rate_Pct.n + Cost..USD....per.Kg.n + Delay_Days.n + Disruption_Event +
    Route_Type + Product_Category + Transportation_Mode,
  data      = df.train,
  method    = "kknn",
  tuneGrid  = tunegrid,
  trControl = ctrl,
  metric    = "Kappa"
)

print(cv.fit.knn)
plot(cv.fit.knn)

best.param.knn <- cv.fit.knn$bestTune
best.param.knn

best.knn <- kknn(
  Mitigation_Action_Taken ~ Geopolitical_Risk_Index.n + Weather_Severity_Index.n +
    Inflation_Rate_Pct.n + Cost..USD....per.Kg.n + Delay_Days.n + Disruption_Event +
    Route_Type + Product_Category + Transportation_Mode,
  train    = df.train,
  test     = df.test,
  k        = best.param.knn$kmax,
  distance = best.param.knn$distance,
  kernel   = as.character(best.param.knn$kernel)
)

df.test$pred.knn.best <- fitted(best.knn)

table(Predicted = df.test$pred.knn.best, Actual = df.test$Mitigation_Action_Taken)
accuracy.knn.best <- mean(df.test$pred.knn.best == df.test$Mitigation_Action_Taken)
cat("Best kNN Accuracy:", accuracy.knn.best, "\n")

confusionMatrix(df.test$pred.knn.best, df.test$Mitigation_Action_Taken)


The kNN classification model demonstrated that combinations of operational and disruption-related variables could effectively distinguish between different mitigation response patterns.


# Decision Tree Analysis

Decision trees were used to identify the strongest branching variables associated with mitigation actions. This approach improved interpretability by visually displaying the sequence of conditions associated with different corrective supply chain responses.


In [ ]:
tree.base <- rpart(
  Mitigation_Action_Taken ~ Geopolitical_Risk_Index + Weather_Severity_Index +
    Inflation_Rate_Pct + Cost..USD....per.Kg + Delay_Days +
    Disruption_Event + Route_Type + Product_Category + Transportation_Mode,
  data    = df.train,
  method  = "class",
  control = rpart.control(
    cp        = 0.01,
    split     = "gini",
    minsplit  = 20,
    minbucket = 10,
    maxdepth  = 5
  )
)

rpart.plot(tree.base, main = "Base Decision Tree — Delivery Status")
summary(tree.base)

df.test$pred.dt.base <- predict(tree.base, df.test, type = "class")

table(Predicted = df.test$pred.dt.base, Actual = df.test$Mitigation_Action_Taken)
accuracy.dt.base <- mean(df.test$pred.dt.base == df.test$Mitigation_Action_Taken)
cat("Base Decision Tree Accuracy:", accuracy.dt.base, "\n")

confusionMatrix(df.test$pred.dt.base, df.test$Mitigation_Action_Taken)

ctrl <- trainControl(method = "cv", number = 5)

tunegrid <- expand.grid(
  cp = seq(0.001, 0.050, by = 0.005)
)

set.seed(42)
cv.fit.dt <- train(
  Mitigation_Action_Taken ~ Geopolitical_Risk_Index + Weather_Severity_Index +
    Inflation_Rate_Pct + Cost..USD....per.Kg + Delay_Days +
    Disruption_Event + Route_Type + Product_Category + Transportation_Mode,
  data      = df.train,
  method    = "rpart",
  tuneGrid  = tunegrid,
  trControl = ctrl,
  control   = rpart.control(split = "gini"),
  metric    = "Kappa"
)

print(cv.fit.dt)
plot(cv.fit.dt, main = "CV Kappa by Complexity Parameter (cp)")

best.param.dt <- cv.fit.dt$bestTune
best.param.dt

tree.best <- rpart(
  Mitigation_Action_Taken ~ Geopolitical_Risk_Index + Weather_Severity_Index +
    Inflation_Rate_Pct + Cost..USD....per.Kg + Delay_Days +
    Disruption_Event + Route_Type + Product_Category + Transportation_Mode,
  data    = df.train,
  method  = "class",
  control = rpart.control(
    cp    = best.param.dt$cp,
    split = "gini"
  )
)

rpart.plot(tree.best, main = "Best Decision Tree — Corrective Action")
summary(tree.best)

df.test$pred.dt.best <- predict(tree.best, df.test, type = "class")

table(Predicted = df.test$pred.dt.best, Actual = df.test$Mitigation_Action_Taken)
accuracy.dt.best <- mean(df.test$pred.dt.best == df.test$Mitigation_Action_Taken)
cat("Best Decision Tree Accuracy:", accuracy.dt.best, "\n")

confusionMatrix(df.test$pred.dt.best, df.test$Mitigation_Action_Taken)


The decision tree analysis revealed that disruption event and shipment industry were among the strongest predictors of which mitigation actions organizations implemented during shipment disruptions.


# Model Summary Comparison

The final comparison evaluates how effectively the kNN and decision tree models classified mitigation actions across the dataset.


In [ ]:
knn.cm  <- confusionMatrix(df.test$pred.knn.best, df.test$Mitigation_Action_Taken)
base.cm <- confusionMatrix(df.test$pred.dt.base,  df.test$Mitigation_Action_Taken)
best.cm <- confusionMatrix(df.test$pred.dt.best,  df.test$Mitigation_Action_Taken)

cat("\n=== Best kNN Model Performance ===\n")
cat("Accuracy:    ", knn.cm$overall['Accuracy'],    "\n")
cat("Kappa:       ", knn.cm$overall['Kappa'],       "\n")
cat("Sensitivity: ", knn.cm$byClass['Sensitivity'], "\n")
cat("Specificity: ", knn.cm$byClass['Specificity'], "\n")

cat("\n=== Base Decision Tree Performance ===\n")
cat("Accuracy:    ", base.cm$overall['Accuracy'],    "\n")
cat("Kappa:       ", base.cm$overall['Kappa'],       "\n")
cat("Sensitivity: ", base.cm$byClass['Sensitivity'], "\n")
cat("Specificity: ", base.cm$byClass['Specificity'], "\n")

cat("\n=== Best Decision Tree Performance ===\n")
cat("Accuracy:    ", best.cm$overall['Accuracy'],    "\n")
cat("Kappa:       ", best.cm$overall['Kappa'],       "\n")
cat("Sensitivity: ", best.cm$byClass['Sensitivity'], "\n")
cat("Specificity: ", best.cm$byClass['Specificity'], "\n")

comparison <- data.frame(
  Model       = c("Best kNN", "Base Decision Tree", "Best Decision Tree"),
  Accuracy    = c(knn.cm$overall['Accuracy'], base.cm$overall['Accuracy'], best.cm$overall['Accuracy']),
  Kappa       = c(knn.cm$overall['Kappa'], base.cm$overall['Kappa'], best.cm$overall['Kappa']),
  Sensitivity = c(knn.cm$byClass['Sensitivity'], base.cm$byClass['Sensitivity'], best.cm$byClass['Sensitivity']),
  Specificity = c(knn.cm$byClass['Specificity'], base.cm$byClass['Specificity'], best.cm$byClass['Specificity'])
)

print(comparison)


The final comparison showed that both the kNN and decision tree models successfully identified meaningful patterns associated with corrective supply chain responses.


# Summary

- Disruption event emerged as one of the strongest predictors of mitigation actions taken during shipment delays.
- Geopolitical conflict-related disruptions were strongly associated with more intensive corrective supply chain responses.
- Route type and transportation mode also contributed to mitigation decision patterns across the dataset.
- The decision tree model improved interpretability by visually identifying the operational conditions associated with different mitigation actions.
- The kNN model demonstrated that combinations of operational and disruption-related variables collectively influenced corrective supply chain decisions.
- Overall, the analysis showed that disruption characteristics and operational routing factors played a major role in determining how organizations responded to shipment delays.
